In [ ]:
from utils_sm import *
from utils_SDEMEM_realdata import *
import os
os.environ['KMP_DUPLICATE_LIB_OK'] = 'True'
import numpy as np
import torch
import math
import random
import matplotlib.pyplot as plt
import torch.optim as optim
from tqdm import tqdm
import time

In [ ]:
obs_size = 40

In [ ]:
# ========== load observed data
df = pd.read_excel("realdata/20160427_mean_eGFP.xlsx", header=None)
x_obs = torch.tensor(df.to_numpy(), dtype=torch.float32)[:, 1:].T.log() 

x_obs = x_obs[:obs_size]

In [ ]:
# ===== Setting for real data ===== #
var_name = ['log_m0', 'log_scale', 'log_offset', 'log_sigma', 'mu_delta', 'mu_gamma', 'mu_k', 'mu_t0', 'log_tau_delta', 'log_tau_gamma', 'log_tau_k', 'log_tau_t0']

# The first 8 parameters have normal priors
prior_mean = torch.tensor([5, 1, 3, -1, -1, -5, 0.5, 0], dtype = torch.float32).unsqueeze(0).to(device)
prior_std = torch.tensor([1, 1, 1, 1, 1, 2, 1, 1], dtype = torch.float32).unsqueeze(0).to(device)

# For the last 4 parameters, the precision follows Gamma priors
prior_alpha = torch.tensor([2, 2, 2, 2], dtype = torch.float32).unsqueeze(0).to(device)
prior_beta = torch.tensor([0.5, 0.5, 0.5, 0.5], dtype = torch.float32).unsqueeze(0).to(device)

T = 30
theta_dim = 12
x_dim = 180 

In [ ]:
theta_ours = torch.from_numpy(np.load(f"sample_res_all/theta_post_precond_mchain_sameprior_task{0}.npy"))

In [ ]:
from sklearn.cluster import KMeans

labels = KMeans(n_clusters=2, random_state=0, n_init="auto").fit_predict(
    theta_ours[:, [4, 5]].detach().cpu().numpy()
)

idx_cluster1 = labels == 0
idx_cluster2 = labels == 1

In [ ]:
idx_cluster1.sum(), idx_cluster2.sum()

In [ ]:
idx_mode1 = idx_cluster1

In [ ]:
x_simu_ours_mode1 = gen_x_given_theta(theta_ours[idx_mode1].repeat(10, 1), T = 30, epis = 0.01, len_interpolate = 1/6, seed = 42, mute = True, use_soft_Ind = False)
x_simu_ours_mode2 = gen_x_given_theta(theta_ours[~idx_mode1].repeat(5, 1), T = 30, epis = 0.01, len_interpolate = 1/6, seed = 42, mute = True, use_soft_Ind = False)

### Band

In [ ]:
q_low_ours_mode1 = torch.quantile(x_simu_ours_mode1, 0.025, dim=0)
q_med_ours_mode1 = torch.quantile(x_simu_ours_mode1, 0.5, dim=0)
q_high_ours_mode1 = torch.quantile(x_simu_ours_mode1, 0.975, dim=0)

q_low_ours_mode2 = torch.quantile(x_simu_ours_mode2, 0.025, dim=0)
q_med_ours_mode2 = torch.quantile(x_simu_ours_mode2, 0.5, dim=0)
q_high_ours_mode2 = torch.quantile(x_simu_ours_mode2, 0.975, dim=0)



p = x_simu_ours_mode1.shape[1]
t = torch.linspace(0, 30, p).cpu().numpy()
x_obs_np = x_obs[:obs_size].T.cpu().numpy()

fig, axes = plt.subplots(1, 2, figsize=(12, 4), dpi=300, sharey=True)

plot_data = [
    (axes[0], q_low_ours_mode1, q_med_ours_mode1, q_high_ours_mode1, "Mode 1"),
    (axes[1], q_low_ours_mode2, q_med_ours_mode2, q_high_ours_mode2, "Mode 2"),
]

for ax, q_low, q_med, q_high, title in plot_data:
    # Plot observed time series
    ax.plot(
        t,
        x_obs_np,
        color="0.55",
        alpha=0.35,
        linewidth=0.8,
        zorder=1
    )

    # Plot posterior predictive interval
    ax.fill_between(
        t,
        q_low.cpu().numpy(),
        q_high.cpu().numpy(),
        color="tab:blue",
        alpha=0.35,
        zorder=2
    )

    # Plot posterior predictive median
    ax.plot(
        t,
        q_med.cpu().numpy(),
        color="tab:blue",
        linewidth=2.0,
        zorder=3
    )

    ax.set_title(title, fontsize=16)
    ax.set_xlabel("Time", fontsize=16)
    ax.set_xlim(-0.5, 30.5)
    ax.set_xticks([0, 5, 10, 15, 20, 25, 30])
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)
    ax.tick_params(axis="both", labelsize=14)
    ax.grid(False)

axes[0].set_ylabel(r"$y(t)$", fontsize=16)

# Create shared legend handles
obs_handle = plt.Line2D(
    [0], [0],
    color="0.55",
    alpha=0.35,
    linewidth=1.2,
    label="Observed trajectories"
)
band_handle = plt.Rectangle(
    (0, 0), 1, 1,
    color="tab:blue",
    alpha=0.35,
    label="95% posterior predictive band"
)
med_handle = plt.Line2D(
    [0], [0],
    color="tab:blue",
    linewidth=2.0,
    label="Posterior predictive median trajectory"
)

# Place one shared legend above all subplots
fig.legend(
    handles=[obs_handle, band_handle, med_handle],
    loc="lower center",
    ncol=3,
    frameon=False,
    fontsize=16,
    bbox_to_anchor=(0.5, 0.0)
)

fig.tight_layout(rect=[0, 0.12, 1, 1])
fig.savefig("Figs/SDE_realdata_yt_mode12.pdf", bbox_inches="tight")

plt.show()